In [1]:
import sounddevice as sd

print(sd.query_devices())

import sounddevice as sd
import queue
import json
import time

from vosk import Model, KaldiRecognizer
import Arm_Lib

Arm = Arm_Lib.Arm_Device()

HOME = [90,135,0,0,90,10]

Arm.Arm_serial_servo_write6_array(HOME,1000)

def home():
    print("Home")
    Arm.Arm_serial_servo_write6_array(HOME,1000)


def open_gripper():
    print("Open Gripper")
    Arm.Arm_serial_servo_write(6,60,500)


def close_gripper():
    print("Close Gripper")
    Arm.Arm_serial_servo_write(6,180,500)
    
def wave():

    print("Wave")

    pose1=[90,90,45,45,90,10]
    pose2=[90,90,45,45,130,10]
    pose3=[90,90,45,45,50,10]

    Arm.Arm_serial_servo_write6_array(pose1,800)

    for i in range(3):

        Arm.Arm_serial_servo_write6_array(pose2,500)
        time.sleep(0.6)

        Arm.Arm_serial_servo_write6_array(pose3,500)
        time.sleep(0.6)

    home()
    
def dance():
    print("Dancing...")

    Arm.Arm_serial_servo_write6(90, 90, 90, 90, 90, 90, 500)
    time.sleep(1)

    # Up
    Arm.Arm_serial_servo_write(2, 60, 500)
    Arm.Arm_serial_servo_write(3, 120, 500)
    Arm.Arm_serial_servo_write(4, 60, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 45, 500)
    Arm.Arm_serial_servo_write(3, 135, 500)
    Arm.Arm_serial_servo_write(4, 45, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 60, 500)
    Arm.Arm_serial_servo_write(3, 120, 500)
    Arm.Arm_serial_servo_write(4, 60, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 90, 500)
    Arm.Arm_serial_servo_write(3, 90, 500)
    Arm.Arm_serial_servo_write(4, 90, 500)
    time.sleep(0.5)

    # Down
    Arm.Arm_serial_servo_write(2, 100, 500)
    Arm.Arm_serial_servo_write(3, 80, 500)
    Arm.Arm_serial_servo_write(4, 80, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 120, 500)
    Arm.Arm_serial_servo_write(3, 60, 500)
    Arm.Arm_serial_servo_write(4, 60, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 135, 500)
    Arm.Arm_serial_servo_write(3, 45, 500)
    Arm.Arm_serial_servo_write(4, 45, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(2, 90, 500)
    Arm.Arm_serial_servo_write(3, 90, 500)
    Arm.Arm_serial_servo_write(4, 90, 500)
    time.sleep(0.5)

    # Wrist dance
    for _ in range(2):
        Arm.Arm_serial_servo_write(4, 20, 500)
        Arm.Arm_serial_servo_write(6, 150, 500)
        time.sleep(0.5)

        Arm.Arm_serial_servo_write(4, 90, 500)
        Arm.Arm_serial_servo_write(6, 90, 500)
        time.sleep(0.5)

    # Side pose
    Arm.Arm_serial_servo_write(1, 0, 500)
    Arm.Arm_serial_servo_write(5, 0, 500)
    time.sleep(0.5)

    # Final spin
    Arm.Arm_serial_servo_write(3, 180, 500)
    Arm.Arm_serial_servo_write(4, 0, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(6, 180, 500)
    time.sleep(0.5)

    Arm.Arm_serial_servo_write(6, 0, 1000)
    time.sleep(0.5)

    # Return home
    Arm.Arm_serial_servo_write6(90,135,0,0,90,10,1000)
    time.sleep(1)

    print("Dance complete!")

def left_right():

    print("Left Right")

    for i in range(2):

        Arm.Arm_serial_servo_write(1,50,400)
        time.sleep(0.5)

        Arm.Arm_serial_servo_write(1,130,400)
        time.sleep(0.5)

    home()
    
def rotate():
    print("rotate...")

    Arm.Arm_serial_servo_write6(90, 90, 90, 90, 90, 90, 500)
    time.sleep(1)

    # rotate
    Arm.Arm_serial_servo_write(1, 0, 500)
    
def bend():
    print("bend...")
    
    Arm.Arm_serial_servo_write6(90, 90, 90, 90, 90, 90, 500)
    time.sleep(1)

    Arm.Arm_serial_servo_write(3,45,800)
    time.sleep(1)
       
def NO():
    print("no.....")

    for i in range(3):
        Arm.Arm_serial_servo_write(1, 75, 300)
        time.sleep(0.3)

        Arm.Arm_serial_servo_write(1, 105, 300)
        time.sleep(0.3)

        Arm.Arm_serial_servo_write(1, 90, 500)
    
def Yes():
    print("Yes...")

    Arm.Arm_serial_servo_write(2, 90, 1000)
    time.sleep(1.5)

    Arm.Arm_serial_servo_write(3, 90, 1000)
    time.sleep(1.5)

    Arm.Arm_serial_servo_write(3, 70, 800)
    time.sleep(1)

for i in range(3):
    Arm.Arm_serial_servo_write(4, 75, 300)
    time.sleep(0.4)
    
    Arm.Arm_serial_servo_write(4, 105, 300)
    time.sleep(0.4)

# Return to center
Arm.Arm_serial_servo_write(4, 90, 500)


# Voice Setup
model = Model("vosk-model-small-en-us-0.15")

q = queue.Queue()

def callback(indata, frames, time, status):
    q.put(bytes(indata))

mic_index = 3
device_info = sd.query_devices(mic_index, 'input')
samplerate = int(device_info['default_samplerate'])

rec = KaldiRecognizer(
    model,
    samplerate,
    '["home", "open", "wave", "close", "dance", "left right","rotate","bend","no","yes" ,"reset", "exit" ]'
)

running = True

last_command = ""

def execute_command(text):

    global running
    global last_command

    text = text.lower().strip()

    if text == last_command:
        return

    last_command = text

    print("Command :", text)

    if "home" in text:
        home()

    elif "open" in text:
        open_gripper()

    elif "close" in text:
        close_gripper()

    elif "wave" in text:
        wave()

    elif "dance" in text:
        dance()

    elif "left right" in text:
        left_right()

    elif "stop" in text:
        home()
        
    elif "rotate" in text:
        rotate()
        
    elif "bend" in text:
        bend()
    
    elif "no" in text:
        NO()
    
    elif "yes" in text:
        Yes()
        
    elif "exit" in text:
        print("Stopping...")
        home()
        running = False

    else:
        print("Unknown command")
        
with sd.RawInputStream(
    device=mic_index,
    samplerate=samplerate,
    blocksize=4096,
    dtype='int16',
    channels=1,
    callback=callback):
    print("----------------------------------")
    print("Voice Control Started")
    print("----------------------------------")
    print("Commands:")
    print("home")
    print("open")
    print("close")
    print("wave")
    print("dance")
    print("rotate")
    print("bend")
    print("left right")
    print("no")
    print("Yes")
    print("stop")
    print("exit")
    print("----------------------------------")

    while running:
        data = q.get()

        if rec.AcceptWaveform(data):
            result = json.loads(rec.Result())
            text = result.get("text", "")

            if text:
                print("🧠 Heard:", text)
                execute_command(text)

print("Listener stopped")
home()


   0 bcm2835 HDMI 1: - (hw:0,0), ALSA (0 in, 8 out)
   1 bcm2835 Headphones: - (hw:1,0), ALSA (0 in, 8 out)
   2 USB 2.0 Camera: Audio (hw:2,0), ALSA (1 in, 0 out)
   3 USB PnP Audio Device: Audio (hw:3,0), ALSA (2 in, 2 out)
   4 sysdefault, ALSA (0 in, 128 out)
   5 samplerate, ALSA (0 in, 128 out)
   6 speexrate, ALSA (0 in, 128 out)
   7 pulse, ALSA (32 in, 32 out)
   8 upmix, ALSA (0 in, 8 out)
   9 vdownmix, ALSA (0 in, 6 out)
  10 dmix, ALSA (0 in, 2 out)
* 11 default, ALSA (32 in, 32 out)
  12 /dev/dsp, OSS (0 in, 16 out)
  13 /dev/dsp1, OSS (0 in, 16 out)
  14 /dev/dsp2, OSS (16 in, 0 out)
  15 /dev/dsp3, OSS (16 in, 16 out)


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-small-en-us-0.15/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from vosk-model-small-en-us-0.15/graph/HCLr.fst vosk-model-small-en-us-0.15/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-small-en-us-0.15/graph/phones/word_boundary.int
LOG (VoskAPI:UpdateGrammarFst():recognizer.cc:287) ["home", "open", "wave", "close", "dance", "left 

----------------------------------
Voice Control Started
----------------------------------
Commands:
home
open
close
wave
dance
rotate
bend
left right
no
Yes
stop
exit
----------------------------------
🧠 Heard: wave
Command : wave
Wave
Home
🧠 Heard: bend
Command : bend
bend...
🧠 Heard: dance
Command : dance
Dancing...
Dance complete!
🧠 Heard: left right
Command : left right
Left Right
Home
🧠 Heard: exit exit
Command : exit exit
Stopping...
Home
Listener stopped
Home
